In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"D:\Drone-research")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SELECTED_LOG = (
    RAW_DIR / "2026-07-02 18-37-42.bin"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.log_parser import parse_flight_log

# Extract only the telemetry needed today
data = parse_flight_log(
    SELECTED_LOG,
    message_types=["ATT", "GPS", "BARO"]
)

# Keep only primary GPS and barometer instances
att_df = data["ATT"].copy()
gps_df = data["GPS"].query("I == 0").copy()
baro_df = data["BARO"].query("I == 0").copy()

# Establish one common time origin
time_origin = min(
    att_df["TimeUS"].min(),
    gps_df["TimeUS"].min(),
    baro_df["TimeUS"].min()
)

for df in [att_df, gps_df, baro_df]:
    df["Time_s"] = (
        df["TimeUS"] - time_origin
    ) / 1_000_000

print("Common time origin:", time_origin, "microseconds")

print("\nLoaded telemetry:")
print("ATT records:", len(att_df))
print("GPS records:", len(gps_df))
print("BARO records:", len(baro_df))

print("\nTime ranges (seconds):")
for name, df in [
    ("ATT", att_df),
    ("GPS", gps_df),
    ("BARO", baro_df)
]:
    print(
        f"{name}: "
        f"{df['Time_s'].min():.2f} - "
        f"{df['Time_s'].max():.2f}"
    )


Common time origin: 23844996 microseconds

Loaded telemetry:
ATT records: 11437
GPS records: 5718
BARO records: 11437

Time ranges (seconds):
ATT: 0.00 - 1301.30
GPS: 0.10 - 1301.14
BARO: 0.00 - 1301.30


In [2]:

import numpy as np
import pandas as pd

telemetry = {
    "ATT": att_df,
    "GPS": gps_df,
    "BARO": baro_df
}

gap_results = []

for name, df in telemetry.items():

    # Sort timestamps before calculating intervals
    times = np.sort(df["TimeUS"].to_numpy())

    # Convert microsecond differences to milliseconds
    intervals_ms = np.diff(times) / 1000

    # Consider only positive intervals
    intervals_ms = intervals_ms[intervals_ms > 0]

    if len(intervals_ms) == 0:
        continue

    median_interval = np.median(intervals_ms)
    maximum_interval = np.max(intervals_ms)

    # Flag intervals greater than 3x the median
    gap_threshold = 3 * median_interval

    large_gaps = intervals_ms > gap_threshold

    gap_results.append({
        "Sensor": name,
        "Median_Interval_ms": round(median_interval, 2),
        "Max_Interval_ms": round(maximum_interval, 2),
        "Gap_Threshold_ms": round(gap_threshold, 2),
        "Large_Gap_Count": int(large_gaps.sum())
    })

gap_df = pd.DataFrame(gap_results)

display(gap_df)


,Sensor,Median_Interval_ms,Max_Interval_ms,Gap_Threshold_ms,Large_Gap_Count
0,ATT,100.0,65500.60,300.00,4
1,GPS,200.0,65560.13,600.01,4
2,BARO,100.0,65500.49,300.00,4


In [3]:

import numpy as np
import pandas as pd

# Build a common timeline at 10 Hz
START_US = int(time_origin)

END_US = int(max(
    att_df["TimeUS"].max(),
    gps_df["TimeUS"].max(),
    baro_df["TimeUS"].max()
))

timeline = pd.DataFrame({
    "TimeUS": np.arange(
        START_US,
        END_US + 1,
        100_000,
        dtype=np.int64
    )
})

# Select fields for synchronization
att_sync = att_df[
    ["TimeUS", "Roll", "Pitch", "Yaw"]
].sort_values("TimeUS")

gps_sync = gps_df[
    ["TimeUS", "Status", "Spd", "Alt"]
].rename(columns={
    "Alt": "GPS_Alt"
}).sort_values("TimeUS")

baro_sync = baro_df[
    ["TimeUS", "Alt", "Press"]
].rename(columns={
    "Alt": "BARO_Alt"
}).sort_values("TimeUS")

# Match ATT values within 150 ms
synced_df = pd.merge_asof(
    timeline,
    att_sync,
    on="TimeUS",
    direction="nearest",
    tolerance=150_000
)

# Match GPS values within 250 ms
synced_df = pd.merge_asof(
    synced_df,
    gps_sync,
    on="TimeUS",
    direction="nearest",
    tolerance=250_000
)

# Match BARO values within 150 ms
synced_df = pd.merge_asof(
    synced_df,
    baro_sync,
    on="TimeUS",
    direction="nearest",
    tolerance=150_000
)

# Convert to elapsed seconds
synced_df["Time_s"] = (
    synced_df["TimeUS"] - START_US
) / 1_000_000

print("Synchronized records:", len(synced_df))

print("\nMissing values after synchronization:")
print(
    synced_df[
        ["Roll", "Pitch", "Yaw", "Spd", "BARO_Alt"]
    ].isna().sum()
)

display(synced_df.head())


Synchronized records: 13014

Missing values after synchronization:
Roll        1569
Pitch       1569
Yaw         1569
Spd         1563
BARO_Alt    1569
dtype: int64


,TimeUS,Roll,Pitch,Yaw,Status,Spd,GPS_Alt,BARO_Alt,Press,Time_s
0,23844996,1.24,-0.47,0.03,3.0,0.018,222.71,0.383432,97617.000000,0.0
1,23944996,1.24,-0.47,0.02,3.0,0.018,222.71,0.405660,97616.750000,0.1
2,24044996,1.24,-0.47,0.02,3.0,0.018,222.71,0.372318,97617.125000,0.2
3,24144996,1.24,-0.47,0.02,3.0,0.048,222.70,0.369540,97617.140625,0.3
4,24244996,1.25,-0.47,0.02,3.0,0.048,222.70,0.405660,97616.750000,0.4


In [4]:

# Validate synchronized telemetry

print("Total synchronized records:", len(synced_df))

print("\nMissing values per column:")
print(synced_df.isna().sum())

print("\nPercentage of missing values:")
print(
    (synced_df.isna().mean() * 100)
    .round(2)
)

# Identify timeline rows where sensor measurements
# are unavailable across all three streams
missing_all = synced_df[
    synced_df[["Roll", "Spd", "BARO_Alt"]]
    .isna()
    .all(axis=1)
]

print(
    "\nRows missing ATT, GPS and BARO:",
    len(missing_all)
)

print("\nExample missing rows:")
display(missing_all.head())


Total synchronized records: 13014

Missing values per column:
TimeUS         0
Roll        1569
Pitch       1569
Yaw         1569
Status      1563
Spd         1563
GPS_Alt     1563
BARO_Alt    1569
Press       1569
Time_s         0
dtype: int64

Percentage of missing values:
TimeUS       0.00
Roll        12.06
Pitch       12.06
Yaw         12.06
Status      12.01
Spd         12.01
GPS_Alt     12.01
BARO_Alt    12.06
Press       12.06
Time_s       0.00
dtype: float64

Rows missing ATT, GPS and BARO: 1563

Example missing rows:


,TimeUS,Roll,Pitch,Yaw,Status,Spd,GPS_Alt,BARO_Alt,Press,Time_s
418,65644996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.8
419,65744996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.9
420,65844996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.0
421,65944996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.1
422,66044996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42.2


In [5]:

import pandas as pd

# Work with a copy of our synchronized dataset
segmented_df = synced_df.copy()

# Check whether the essential measurements are available
segmented_df["Data_Available"] = (
    segmented_df[["Roll", "Spd", "BARO_Alt"]]
    .notna()
    .all(axis=1)
)

# Assign a new segment whenever availability changes
segmented_df["Segment_ID"] = (
    segmented_df["Data_Available"]
    .ne(segmented_df["Data_Available"].shift())
    .cumsum()
)

# Summarize continuous regions
segment_summary = (
    segmented_df.groupby("Segment_ID")
    .agg(
        Start_s=("Time_s", "min"),
        End_s=("Time_s", "max"),
        Rows=("Time_s", "size"),
        Data_Available=("Data_Available", "first")
    )
    .reset_index()
)

segment_summary["Duration_s"] = (
    segment_summary["End_s"]
    - segment_summary["Start_s"]
    + 0.1
).round(2)

display(segment_summary)


,Segment_ID,Start_s,End_s,Rows,Data_Available,Duration_s
0,1,0.0,41.7,418,True,41.8
1,2,41.8,62.1,204,False,20.4
2,3,62.2,543.4,4813,True,481.3
3,4,543.5,608.6,652,False,65.2
4,5,608.7,912.2,3036,True,303.6
5,6,912.3,941.9,297,False,29.7
6,7,942.0,1164.1,2222,True,222.2
7,8,1164.2,1205.7,416,False,41.6
8,9,1205.8,1301.3,956,True,95.6


In [6]:

from pathlib import Path

# Create processed data directory
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

log_id = SELECTED_LOG.stem.replace(" ", "_")

# Output paths
sync_path = PROCESSED_DIR / f"{log_id}_synchronized.csv"

segment_path = (
    PROCESSED_DIR / f"{log_id}_segments.csv"
)

# Save synchronized telemetry with segment information
segmented_df.to_csv(
    sync_path,
    index=False
)

# Save availability segment summary
segment_summary.to_csv(
    segment_path,
    index=False
)

print("Synchronized dataset saved:", sync_path.name)
print("Availability summary saved:", segment_path.name)

print("\nSynchronized records:", len(segmented_df))
print("Availability regions:", len(segment_summary))


Synchronized dataset saved: 2026-07-02_18-37-42_synchronized.csv
Availability summary saved: 2026-07-02_18-37-42_segments.csv

Synchronized records: 13014
Availability regions: 9
